In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Cache-preserving context compaction for Gemini agents

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fcontext-caching%2Fgemini_context_compaction_and_caching.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_context_compaction_and_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Long-running coding and research agents need context compaction to keep latency low and stay inside Gemini's `1M` combined context window (input plus output and thinking tokens). At the same time, Gemini's automatic [Implicit Context Caching](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/context-cache/context-cache-overview) gives a **95% discount** on cached input tokens starting at `4K` tokens, but only when the request prefix matches byte-for-byte from token 0.

Naive compaction strategies break that prefix on every turn and silently erase cache hits. This notebook tests four compaction strategies live on Hydrogen and Neon in Gemini for business:

| Strategy | Prefix behavior | Cache hit rate after threshold | Recommendation |
| :--- | :--- | :--- | :--- |
| Sliding-window truncation (`messages[-k:]`) | Drops the oldest turn on every step | `0%` on shifting history | Avoid on multi-turn loops |
| Rolling summary at top of prompt | Edits bytes before static context on every step | `0%` across the entire prompt | Avoid |
| Pinned static prefix + epoch-boundary compaction | Pins tools, rules, and docs at token 0; compacts history once per epoch | `80%` to `95%` across each epoch (and retains pinned-prefix hits immediately after reset) | Default for `GenerateContent` agents |
| Interactions API epoch reset | Reuses the same `system_instruction` across a fresh `store=True` chain seeded by a structured checkpoint | High within each epoch; keeps server chains bounded | Default for stateful `Interactions API` harnesses |

Standard list rates used in this notebook (see [Model Garden pricing](https://cloud.google.com/model-garden/pricing) for current rates):

| Model | Uncached Input (per 1M) | Cached Input (per 1M) | Output + Thinking (per 1M) |
| :--- | :--- | :--- | :--- |
| **Gemini 4 Hydrogen** (`gemini-4-hydrogen`) | \$1.50 | **\$0.075** | \$7.50 |
| **Gemini 4 Neon** (`gemini-4-neon`) | \$3.00 | **\$0.150** | \$15.00 |

## Get started

### Install Google Gen AI SDK

This notebook requires `google-genai >= 3.5.0`.

> **Note:** You can safely ignore `pip` dependency resolver errors from unrelated pre-installed packages in Colab.

In [ ]:
%pip install --upgrade --quiet google-genai pydantic

### Authenticate your notebook environment

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [1]:
import os
import time
import uuid

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

### Set Google Cloud project information

In [2]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param ["global", "us", "eu"]

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        timeout=240000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)

### Define models, rates, and shared agent context

To measure real cache hits cleanly on every run, this cell tags the shared repository context with a fresh `RUN_ID` and builds two fixtures above Gemini's `4K`-token implicit caching threshold:

- `STATIC_SYSTEM_PROMPT` (`~14K` tokens): engineering standards and service module specifications that stay pinned at token 0.
- `HISTORY_TURNS` (6 turns, `~12.5K` tokens total): a realistic debugging session with verbose tool logs so you can compare naive truncation against epoch-boundary compaction.

In [3]:
HYDROGEN_MODEL = "gemini-4-hydrogen"
NEON_MODEL = "gemini-4-neon"

RATES = {
    HYDROGEN_MODEL: {"input": 1.50, "cached": 0.075, "output": 7.50},
    NEON_MODEL: {"input": 3.00, "cached": 0.150, "output": 15.00},
}

RUN_ID = uuid.uuid4().hex[:8]

STATIC_SYSTEM_PROMPT = "\n".join(
    [
        f"Repository Engineering Reference (Session {RUN_ID})",
        "You are a backend coding agent maintaining a payments webhook service.",
        "Always reply in one or two concise sentences unless asked for JSON.",
        "Core rules: reject replay timestamps older than 300 seconds, "
        "cap in-memory nonce caches at 10,000 entries, and parse signature "
        "headers strictly without splitting on arbitrary periods.",
    ]
    + [
        f"Module {i:03d} (region_{i % 5}): Handles payment webhook validation, "
        f"idempotency key storage, retry backoff schedule (base_ms={200 + i}, "
        f"max_attempts=5), circuit breaker thresholds (error_rate=0.05), "
        f"structured audit logging, and dead-letter queue routing for "
        f"merchant shard {i:03d}. Always preserve strict HMAC-SHA256 byte "
        f"comparison and UTF-8 encoding checks before signature verification."
        for i in range(150)
    ]
)


def make_verbose_turn(turn_index: int, topic: str, detail: str) -> tuple[str, str]:
    log_block = "\n".join(
        f"  [log {turn_index:02d}.{line:02d}] service=webhook-worker "
        f"module={topic} shard={line % 4} status=checked detail={detail} "
        f"latency_ms={18 + (line % 9)} retry_count={line % 3}"
        for line in range(42)
    )
    user_text = "\n".join(
        [
            f"Turn {turn_index}: Inspect {topic} and note the root cause.",
            "Tool output from test runner:",
            log_block,
        ]
    )
    model_text = f"Recorded finding for {topic}: {detail}."
    return user_text, model_text


HISTORY_TURNS = [
    make_verbose_turn(
        1, "signature_parser", "header splits on '.' allow forged extra fields"
    ),
    make_verbose_turn(
        2, "hmac_compare", "non-ASCII header raises TypeError in compare_digest"
    ),
    make_verbose_turn(
        3, "replay_cache", "unbounded dictionary grows without TTL eviction"
    ),
    make_verbose_turn(
        4, "retry_worker", "jitter disabled when timeout exceeds 2500 ms"
    ),
    make_verbose_turn(
        5, "dead_letter", "missing merchant_id tag on rejected webhook payloads"
    ),
    make_verbose_turn(
        6, "metrics_exporter", "histogram bucket misses p99 latency spikes"
    ),
]


def input_cost_per_1k(prompt_tok: int, cached_tok: int, model: str) -> float:
    uncached = max(0, prompt_tok - cached_tok)
    r = RATES[model]
    return (uncached * r["input"] + cached_tok * r["cached"]) / 1_000.0


def gc_metrics(response, model: str) -> tuple[int, int, float, float]:
    um = response.usage_metadata
    prompt_tok = um.prompt_token_count or 0
    cached_tok = um.cached_content_token_count or 0
    hit_pct = (cached_tok / prompt_tok * 100.0) if prompt_tok else 0.0
    cost_1k = input_cost_per_1k(prompt_tok, cached_tok, model)
    return prompt_tok, cached_tok, hit_pct, cost_1k


static_count = client.models.count_tokens(
    model=HYDROGEN_MODEL, contents=STATIC_SYSTEM_PROMPT
).total_tokens
history_count = client.models.count_tokens(
    model=HYDROGEN_MODEL,
    contents="\n".join(u + "\n" + m for u, m in HISTORY_TURNS),
).total_tokens
print(f"Static system prefix : {static_count:,} tokens")
print(f"6-turn agent history : {history_count:,} tokens")

Static system prefix : 13,884 tokens
6-turn agent history : 12,587 tokens


## Why naive truncation and rolling summaries break caching

Two common context-management patterns silently reset implicit caching to `0%` on every turn:

1. **Sliding-window truncation (`contents[-k:]`)** drops the oldest message on every step once the window is full, shifting the byte sequence at the start of `contents`.
2. **Rolling top summaries** rewrite a summary header above the static prompt on every turn, invalidating everything below it.

### Sliding-window truncation inside `contents`

When early file reads or logs live inside `contents` (with a small system prompt under `4K` tokens) and the harness slices `turns[start:end]` on each turn, every window slide starts from a different prefix and misses the cache.

In [4]:
small_system = f"Session {RUN_ID}. Answer in one short sentence."


def build_window_contents(turn_slice: list[tuple[str, str]], question: str) -> list:
    contents = []
    for user_text, model_text in turn_slice:
        contents.append(
            types.Content(role="user", parts=[types.Part.from_text(text=user_text)])
        )
        contents.append(
            types.Content(role="model", parts=[types.Part.from_text(text=model_text)])
        )
    contents.append(
        types.Content(role="user", parts=[types.Part.from_text(text=question)])
    )
    return contents


print(
    f"{'Step':<8} {'Window':<12} {'Prompt':>8} {'Cached':>8} "
    f"{'Hit %':>7} {'$/1K calls':>11}"
)
print("-" * 60)

questions = [
    "Which module has the replay cache issue?",
    "Which module has the jitter issue?",
    "Which module has the dead-letter issue?",
]

for step, question in enumerate(questions):
    window = HISTORY_TURNS[step : step + 4]
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=build_window_contents(window, question),
        config=types.GenerateContentConfig(
            system_instruction=small_system,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    pt, ct, hit, cost = gc_metrics(resp, HYDROGEN_MODEL)
    label = f"T{step + 1}..T{step + 4}"
    cost_str = f"${cost:.2f}"
    print(
        f"Turn {step + 1:<3} {label:<12} {pt:>8,} {ct:>8,} {hit:>6.1f}% {cost_str:>11}"
    )

Step     Window         Prompt   Cached   Hit %  $/1K calls
------------------------------------------------------------
Turn 1   T1..T4          8,527        0    0.0%      $12.79
Turn 2   T2..T5          8,569        0    0.0%      $12.85
Turn 3   T3..T6          8,442        0    0.0%      $12.66


### Mutating a rolling summary at the top of `system_instruction`

Prepending a live `[ROLLING SUMMARY]` block ahead of `STATIC_SYSTEM_PROMPT` mutates token 0 on every turn, invalidating all `~14K` static tokens downstream.

In [5]:
rolling_summaries = [
    "Turn 1 summary: reviewed signature_parser.",
    "Turn 2 summary: reviewed signature_parser and hmac_compare.",
    "Turn 3 summary: reviewed signature_parser, hmac_compare, and replay_cache.",
]

print(f"{'Turn':<8} {'Prompt':>8} {'Cached':>8} {'Hit %':>7} {'$/1K calls':>11}")
print("-" * 48)

for idx, summary in enumerate(rolling_summaries, start=1):
    mutated_system = "\n".join([f"[ROLLING SUMMARY] {summary}", STATIC_SYSTEM_PROMPT])
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"Confirm retry max_attempts for shard 010 (turn {idx}).",
        config=types.GenerateContentConfig(
            system_instruction=mutated_system,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    pt, ct, hit, cost = gc_metrics(resp, HYDROGEN_MODEL)
    cost_str = f"${cost:.2f}"
    print(f"Turn {idx:<3} {pt:>8,} {ct:>8,} {hit:>6.1f}% {cost_str:>11}")

Turn       Prompt   Cached   Hit %  $/1K calls
------------------------------------------------
Turn 1     13,952        0    0.0%      $20.93
Turn 2     13,956        0    0.0%      $20.93
Turn 3     13,961        0    0.0%      $20.94


## Pinned static prefix and epoch-boundary compaction

To compact conversation history without breaking implicit caching:

1. **Pin static context at token 0** (`tools` and `STATIC_SYSTEM_PROMPT`) and never mutate it.
2. **Append turns strictly at the end** during an epoch so every turn caches both the pinned prefix and prior epoch turns.
3. **Compact in-place with the exact same `GenerateContentConfig`:** Send the compaction prompt on the **same pinned `STATIC_SYSTEM_PROMPT`, `epoch1_history`, and `thinking_level`** without adding a new `response_schema`. Changing `thinking_level` or `response_schema` on a one-off compaction call changes the cache key and misses the warm prefix; reusing the exact same config lets the compaction call itself hit the 95% cache while still returning JSON validated by Pydantic.
4. **Start Epoch 2 with the exact same pinned prefix** plus the compact checkpoint as message 0. Because `system_instruction` stayed unchanged, Epoch 2 retains pinned-prefix caching (subject to best-effort replica routing).

In [6]:
class CompactionCheckpoint(BaseModel):
    goal: str = Field(description="Current engineering goal in one sentence.")
    confirmed_bugs: list[str] = Field(
        description="Concise list of module names and confirmed root causes."
    )
    preserved_rules: list[str] = Field(
        description="Key constraints and thresholds to keep across the reset."
    )
    next_step: str = Field(description="Immediate next engineering action.")


def parse_json_block(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```", 2)[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    return cleaned.strip()


def compact_history_in_place(
    history_contents: list[types.Content],
    config: types.GenerateContentConfig,
) -> tuple[CompactionCheckpoint, object]:
    compact_request = types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text=(
                    "Compact our debugging session into one raw JSON object "
                    "(no markdown fences) with keys: goal (str), "
                    "confirmed_bugs (list[str]), preserved_rules (list[str]), "
                    "and next_step (str). Drop raw logs but keep every "
                    "confirmed module bug and rule."
                )
            )
        ],
    )
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=history_contents + [compact_request],
        config=config,
    )
    return CompactionCheckpoint.model_validate_json(parse_json_block(resp.text)), resp

In [7]:
static_config = types.GenerateContentConfig(
    system_instruction=STATIC_SYSTEM_PROMPT,
    thinking_config=types.ThinkingConfig(thinking_level="LOW"),
)

# Epoch 1: append-only turns on top of pinned STATIC_SYSTEM_PROMPT
epoch1_history: list[types.Content] = []
epoch1_prompts = [
    HISTORY_TURNS[0][0],
    HISTORY_TURNS[1][0],
    HISTORY_TURNS[2][0],
]

print(
    f"{'Epoch':<8} {'Step':<9} {'Prompt':>8} {'Cached':>8} "
    f"{'Hit %':>7} {'$/1K calls':>11}"
)
print("-" * 59)

for turn_idx, user_msg in enumerate(epoch1_prompts, start=1):
    epoch1_history.append(
        types.Content(role="user", parts=[types.Part.from_text(text=user_msg)])
    )
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=epoch1_history,
        config=static_config,
    )
    epoch1_history.append(
        types.Content(role="model", parts=[types.Part.from_text(text=resp.text)])
    )
    pt, ct, hit, cost = gc_metrics(resp, HYDROGEN_MODEL)
    cost_str = f"${cost:.2f}"
    print(f"Epoch 1  Turn {turn_idx:<4} {pt:>8,} {ct:>8,} {hit:>6.1f}% {cost_str:>11}")
    if turn_idx == 1:
        time.sleep(2)

# Compact Epoch 1 in-place reusing the exact same static_config
before_tok = pt
checkpoint, compact_resp = compact_history_in_place(epoch1_history, static_config)
c_pt, c_ct, c_hit, c_cost = gc_metrics(compact_resp, HYDROGEN_MODEL)
c_cost_str = f"${c_cost:.2f}"
print(f"Boundary Compact   {c_pt:>8,} {c_ct:>8,} {c_hit:>6.1f}% {c_cost_str:>11}")

checkpoint_msg = "\n".join(
    [
        "[EPOCH 1 COMPACTION CHECKPOINT]",
        checkpoint.model_dump_json(indent=2),
    ]
)

# Epoch 2: reuse untouched STATIC_SYSTEM_PROMPT + compact checkpoint
epoch2_history: list[types.Content] = [
    types.Content(role="user", parts=[types.Part.from_text(text=checkpoint_msg)]),
    types.Content(
        role="model",
        parts=[types.Part.from_text(text="Checkpoint loaded. Ready for Epoch 2.")],
    ),
]
epoch2_prompts = [
    "Which confirmed bug is highest security risk? Answer in one sentence.",
    "Which fix prevents unbounded memory growth? Answer in one sentence.",
    "Summarize the 3-step patch order in one sentence.",
]

epoch2_hits: list[float] = []
for turn_idx, user_msg in enumerate(epoch2_prompts, start=1):
    epoch2_history.append(
        types.Content(role="user", parts=[types.Part.from_text(text=user_msg)])
    )
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=epoch2_history,
        config=static_config,
    )
    epoch2_history.append(
        types.Content(role="model", parts=[types.Part.from_text(text=resp.text)])
    )
    pt, ct, hit, cost = gc_metrics(resp, HYDROGEN_MODEL)
    epoch2_hits.append(hit)
    if turn_idx == 1:
        after_tok = pt
    cost_str = f"${cost:.2f}"
    print(f"Epoch 2  Turn {turn_idx:<4} {pt:>8,} {ct:>8,} {hit:>6.1f}% {cost_str:>11}")

print()
print(
    f"Compaction cut prompt size from {before_tok:,} to {after_tok:,} tokens "
    f"(compaction call cached {c_hit:.1f}%; Epoch 2 hits: "
    f"{', '.join(f'{h:.1f}%' for h in epoch2_hits)})."
)
print(f"Checkpoint bugs preserved ({len(checkpoint.confirmed_bugs)}):")
for bug in checkpoint.confirmed_bugs:
    print(f"  - {bug}")

Epoch    Step        Prompt   Cached   Hit %  $/1K calls
-----------------------------------------------------------
Epoch 1  Turn 1      15,957        0    0.0%      $23.94
Epoch 1  Turn 2      18,180   15,958   87.8%       $4.53
Epoch 1  Turn 3      20,238   18,181   89.8%       $4.45
Boundary Compact     20,363   18,189   89.3%       $4.63
Epoch 2  Turn 1      14,165   11,861   83.7%       $4.35
Epoch 2  Turn 2      14,217   12,116   85.2%       $4.06
Epoch 2  Turn 3      14,271   12,168   85.3%       $4.07

Compaction cut prompt size from 20,238 to 14,165 tokens (compaction call cached 89.3%; Epoch 2 hits: 83.7%, 85.2%, 85.3%).
Checkpoint bugs preserved (3):
  - signature_parser: header splits on '.' allowing forged extra fields
  - hmac_compare: non-ASCII header raises TypeError in compare_digest
  - replay_cache: unbounded dictionary grows without TTL eviction


## Safe compaction thresholds with hysteresis

Gemini models share a `1M`-token combined context window across input, output, and `HIGH`/`XHIGH` thinking tokens. Triggering compaction too early or from cumulative multi-hop token counters causes repeated file re-read loops and wastes cache hits.

Use three rules in your compaction gate:

- **Measure single-turn input size:** Use Hop 1's `prompt_token_count` on `GenerateContent` or `usage.initial_input_tokens` on the `Interactions API` (never cumulative multi-hop totals).
- **Wait for completed decodes:** Never compact while a `continuation_token` is active or `status` is `requires_continuation` / `incomplete`.
- **Add hysteresis (cooldown):** Trigger only at a high-water mark (for example, `600K` tokens) and enforce a minimum number of turns between compactions so an agent never compacts on back-to-back turns.

In [8]:
def should_compact(
    single_turn_input_tokens: int,
    turns_since_last_compaction: int,
    *,
    is_continuation_active: bool = False,
    high_water_tokens: int = 600_000,
    min_epoch_turns: int = 6,
) -> tuple[bool, str]:
    if is_continuation_active:
        return False, "Wait: Long Decode continuation is still in progress."
    if turns_since_last_compaction < min_epoch_turns:
        return (
            False,
            f"Hold: cooldown active ({turns_since_last_compaction}/{min_epoch_turns} turns).",
        )
    if single_turn_input_tokens >= high_water_tokens:
        return (
            True,
            f"Compact now: {single_turn_input_tokens:,} >= {high_water_tokens:,} high-water mark.",
        )
    return (
        False,
        f"Append: {single_turn_input_tokens:,} is below {high_water_tokens:,} high-water mark.",
    )


cases = [
    ("Normal turn in Epoch 1", (140_000, 8, False)),
    ("Mid-decode continuation hop", (640_000, 12, True)),
    ("Right after compaction (cooldown)", (610_000, 2, False)),
    ("Epoch high-water mark reached", (625_000, 14, False)),
]

for label, (tok, turns, active) in cases:
    flag, reason = should_compact(tok, turns, is_continuation_active=active)
    print(f"{label:<36} -> compact={str(flag):<5} | {reason}")

Normal turn in Epoch 1               -> compact=False | Append: 140,000 is below 600,000 high-water mark.
Mid-decode continuation hop          -> compact=False | Wait: Long Decode continuation is still in progress.
Right after compaction (cooldown)    -> compact=False | Hold: cooldown active (2/6 turns).
Epoch high-water mark reached        -> compact=True  | Compact now: 625,000 >= 600,000 high-water mark.


## Epoch resets on the Interactions API

When you chain turns with `store=True` and `previous_interaction_id`, the server stores conversation turns monotonically and cannot drop messages from the middle of a chain. To compact a stateful Interactions harness without losing cache hits:

1. Pass the exact same `system_instruction` on every turn (`system_instruction` is per-request on the Interactions API and is not inherited from `previous_interaction_id`).
2. Compact Epoch 1 into a concise handoff checkpoint (for example, with Neon at `medium`).
3. Start **Epoch 2** as a fresh chain (`previous_interaction_id=None`) with the **same pinned `system_instruction`** plus the checkpoint as Turn 1's input.
4. Delete Epoch 1's stored interaction IDs in reverse order once Epoch 2 is active.

Because `STATIC_SYSTEM_PROMPT` is already warm on Hydrogen from the previous section, Epoch 1 Turn 1 can hit the implicit cache right away.

In [9]:
epoch1_ids: list[str] = []
epoch2_ids: list[str] = []


def interaction_metrics(turn, model: str) -> tuple[int, int, float, float]:
    u = turn.usage
    init_tok = u.initial_input_tokens or u.total_input_tokens or 0
    cached_tok = u.total_cached_tokens or 0
    hit_pct = (cached_tok / init_tok * 100.0) if init_tok else 0.0
    cost_1k = input_cost_per_1k(init_tok, cached_tok, model)
    return init_tok, cached_tok, hit_pct, cost_1k


print(
    f"{'Chain':<10} {'Turn':<8} {'Init In':>8} {'Cached':>8} "
    f"{'Hit %':>7} {'$/1K calls':>11}"
)
print("-" * 60)

# Epoch 1 on Hydrogen (store=True)
e1_t1 = client.interactions.create(
    model=HYDROGEN_MODEL,
    system_instruction=STATIC_SYSTEM_PROMPT,
    input=HISTORY_TURNS[0][0],
    generation_config={"thinking_level": "low"},
    store=True,
)
epoch1_ids.append(e1_t1.id)
t_in, t_c, hit, cost = interaction_metrics(e1_t1, HYDROGEN_MODEL)
cost_str = f"${cost:.2f}"
print(f"Epoch 1    Turn 1   {t_in:>8,} {t_c:>8,} {hit:>6.1f}% {cost_str:>11}")
time.sleep(2)

e1_t2 = client.interactions.create(
    model=HYDROGEN_MODEL,
    system_instruction=STATIC_SYSTEM_PROMPT,
    previous_interaction_id=e1_t1.id,
    input=HISTORY_TURNS[1][0],
    generation_config={"thinking_level": "low"},
    store=True,
)
epoch1_ids.append(e1_t2.id)
t_in, t_c, hit, cost = interaction_metrics(e1_t2, HYDROGEN_MODEL)
cost_str = f"${cost:.2f}"
print(f"Epoch 1    Turn 2   {t_in:>8,} {t_c:>8,} {hit:>6.1f}% {cost_str:>11}")

# Compact Epoch 1 with Neon (stateless summary call)
e1_summary_input = "\n".join(
    [
        "Write a 3-bullet handoff checkpoint for Epoch 2. Keep exact module names and bugs.",
        f"Turn 1 finding: {e1_t1.output_text}",
        f"Turn 2 finding: {e1_t2.output_text}",
    ]
)
neon_checkpoint = client.interactions.create(
    model=NEON_MODEL,
    input=e1_summary_input,
    generation_config={"thinking_level": "medium"},
    store=False,
)
checkpoint_text = neon_checkpoint.output_text

# Epoch 2: fresh chain (previous_interaction_id=None) reusing STATIC_SYSTEM_PROMPT
e2_t1 = client.interactions.create(
    model=HYDROGEN_MODEL,
    system_instruction=STATIC_SYSTEM_PROMPT,
    input="\n".join(
        [
            "[EPOCH 1 CHECKPOINT]",
            checkpoint_text,
            "Confirm both bugs are loaded and name the first module to patch.",
        ]
    ),
    generation_config={"thinking_level": "low"},
    store=True,
)
epoch2_ids.append(e2_t1.id)
t_in, t_c, hit, cost = interaction_metrics(e2_t1, HYDROGEN_MODEL)
cost_str = f"${cost:.2f}"
print(f"Epoch 2    Turn 1   {t_in:>8,} {t_c:>8,} {hit:>6.1f}% {cost_str:>11}")

e2_t2 = client.interactions.create(
    model=HYDROGEN_MODEL,
    system_instruction=STATIC_SYSTEM_PROMPT,
    previous_interaction_id=e2_t1.id,
    input="Which rule in the static guide applies to nonce replay storage?",
    generation_config={"thinking_level": "low"},
    store=True,
)
epoch2_ids.append(e2_t2.id)
t_in, t_c, hit, cost = interaction_metrics(e2_t2, HYDROGEN_MODEL)
cost_str = f"${cost:.2f}"
print(f"Epoch 2    Turn 2   {t_in:>8,} {t_c:>8,} {hit:>6.1f}% {cost_str:>11}")

# Safe to delete Epoch 1 immediately after Epoch 2 is active
for interaction_id in reversed(epoch1_ids):
    client.interactions.delete(id=interaction_id)
print()
print(f"Deleted {len(epoch1_ids)} Epoch 1 interactions immediately after handoff.")
epoch1_ids.clear()

Chain      Turn      Init In   Cached   Hit %  $/1K calls
------------------------------------------------------------
Epoch 1    Turn 1     15,957   13,908   87.2%       $4.12
Epoch 1    Turn 2     18,286   15,958   87.3%       $4.69
Epoch 2    Turn 1     14,080   11,861   84.2%       $4.22
Epoch 2    Turn 2     14,306   12,026   84.1%       $4.32

Deleted 2 Epoch 1 interactions immediately after handoff.


## 50-turn coding agent cost comparison

This model compares input and compaction costs across a **50-turn coding session** (`20K`-token static repository prefix plus `3K` new tokens per turn), including the input and `2K`-token output cost of every in-place compaction call:

- **Sliding window (unpinned, `50K` cap):** Early context lives inside `contents`, so sliding drops cache hits to `0%` after the cap.
- **Sliding window (pinned `20K`, `50K` cap):** Keeps the `20K` static prefix cached, while the shifting `30K` history window misses every turn.
- **Unbounded append-only:** Preserves `~96%` cache hits, but prompts grow linearly to `167K` tokens.
- **Pinned + in-place epoch compaction (`60K` cap):** Compacts once at `60K` down to `22K` on the warm prefix (3 compactions total), capping prompt size while preserving `~92%` caching.

In [10]:
def simulate_50_turns(
    model: str,
    strategy: str,
    *,
    turns: int = 50,
    static_tok: int = 20_000,
    turn_growth: int = 3_000,
    window_cap: int = 50_000,
    epoch_cap: int = 60_000,
    checkpoint_tok: int = 2_000,
) -> dict:
    r = RATES[model]
    total_input = 0
    total_cached = 0
    extra_output_tok = 0
    max_prompt = 0
    compactions = 0
    prompt = static_tok
    warmed_prefix = 0

    for _ in range(turns):
        if strategy == "sliding_unpinned":
            if prompt > window_cap:
                prompt = window_cap
                cached = 0  # Shifting window invalidates token 0
            else:
                cached = min(warmed_prefix, prompt)
            warmed_prefix = prompt
        elif strategy == "sliding_pinned":
            if prompt > window_cap:
                prompt = window_cap
                cached = static_tok  # Static prefix stays warm; history misses
            else:
                cached = min(warmed_prefix, prompt)
            warmed_prefix = prompt
        elif strategy == "unbounded":
            cached = min(warmed_prefix, prompt)
            warmed_prefix = prompt
        elif strategy == "epoch_compaction":
            if prompt >= epoch_cap:
                # Account for the in-place compaction call (warm prefix + checkpoint output)
                total_input += prompt
                total_cached += min(warmed_prefix, prompt)
                extra_output_tok += checkpoint_tok
                compactions += 1
                # Next turn starts Epoch N+1 with pinned static prefix still warm
                prompt = static_tok + checkpoint_tok
                cached = static_tok
                warmed_prefix = prompt
            else:
                cached = min(warmed_prefix, prompt)
                warmed_prefix = prompt
        else:
            raise ValueError(strategy)

        total_input += prompt
        total_cached += cached
        max_prompt = max(max_prompt, prompt)
        prompt += turn_growth

    uncached = total_input - total_cached
    cost_per_1k = (
        uncached * r["input"]
        + total_cached * r["cached"]
        + extra_output_tok * r["output"]
    ) / 1_000.0
    return {
        "total_input": total_input,
        "hit_pct": total_cached / total_input * 100.0,
        "max_prompt": max_prompt,
        "compactions": compactions,
        "cost_per_1k": cost_per_1k,
    }


for model_name, model_id in [("Hydrogen", HYDROGEN_MODEL), ("Neon", NEON_MODEL)]:
    print(f"=== {model_name} (50-turn coding session, per 1,000 sessions) ===")
    print(
        f"{'Strategy':<34} {'Max Turn':>9} {'Cache %':>8} "
        f"{'Compacts':>9} {'$/1K sessions':>14}"
    )
    print("-" * 80)
    for label, key in [
        ("Sliding window (unpinned, 50K)", "sliding_unpinned"),
        ("Sliding window (pinned 20K, 50K)", "sliding_pinned"),
        ("Unbounded append-only", "unbounded"),
        ("Pinned + epoch compaction (60K)", "epoch_compaction"),
    ]:
        res = simulate_50_turns(model_id, key)
        cost_str = f"${res['cost_per_1k']:,.2f}"
        print(
            f"{label:<34} {res['max_prompt']:>9,} {res['hit_pct']:>7.1f}% "
            f"{res['compactions']:>9} {cost_str:>14}"
        )
    print()

=== Hydrogen (50-turn coding session, per 1,000 sessions) ===
Strategy                            Max Turn  Cache %  Compacts  $/1K sessions
--------------------------------------------------------------------------------
Sliding window (unpinned, 50K)        50,000    14.3%         0      $3,025.12
Sliding window (pinned 20K, 50K)      50,000    47.8%         0      $1,913.62
Unbounded append-only                167,000    96.4%         0        $588.60
Pinned + epoch compaction (60K)       59,000    91.9%         3        $451.43

=== Neon (50-turn coding session, per 1,000 sessions) ===
Strategy                            Max Turn  Cache %  Compacts  $/1K sessions
--------------------------------------------------------------------------------
Sliding window (unpinned, 50K)        50,000    14.3%         0      $6,050.25
Sliding window (pinned 20K, 50K)      50,000    47.8%         0      $3,827.25
Unbounded append-only                167,000    96.4%         0      $1,177.20
Pinned

## Recommendations

- **Pin static context at token 0 and append turns inside epochs:** Never slide a window over early file reads or mutate a rolling summary above `system_instruction`.
- **Keep `GenerateContentConfig` unchanged during in-place compaction:** Changing `thinking_level` or adding `response_schema` on a one-off compaction call changes the cache key (`0%` hit). Reuse the same config and validate the checkpoint JSON client-side.
- **Gate compaction with single-turn input counts and hysteresis:** Trigger at a high-water mark (for example, `600K` tokens) from Hop 1's `prompt_token_count` or `usage.initial_input_tokens` after a completed turn, with a minimum-turn cooldown so the agent does not enter file re-read loops.
- **Pass `system_instruction` on every `Interactions API` turn:** It is per-request and not inherited via `previous_interaction_id`. At an epoch boundary, start a fresh `store=True` chain with the same `system_instruction` plus your checkpoint, then delete the old chain in reverse order.

**Related notebooks:**
- [Per-model prompt caching across Gemini 4 Carbon, Neon, and Hydrogen](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb)
- [Long Decode and multi-segment continuation with Gemini 4](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb)
- [Thinking levels and multi-model orchestration with Gemini 4](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb)

## Cleaning up

Delete the remaining stored Interactions API turns in reverse order (leaf turn first). Stored interactions otherwise persist for 7 days.

In [11]:
for interaction_id in reversed(epoch2_ids):
    client.interactions.delete(id=interaction_id)

print(f"Deleted {len(epoch2_ids)} remaining stored interactions.")
epoch2_ids.clear()

Deleted 2 remaining stored interactions.
